# E1 · Explainable boosting: tuning, calibration and the four-model ensembles

**Explainable boosting** (EBM) is gradient boosting kept additive: it learns one curve per
predictor, and, when the tuning allows, a few curves for pairs of predictors. Its prediction is
the sum of those curves, so every prediction splits exactly across predictors, as logistic
regression's does, but the curves can bend (a distance that matters up to an hour and then
stops mattering, say). This track adds it as the library's fourth model.

Everything is done exactly as for the three core models: the same folds, weights, objective and
held-out women; the same nested calibration; and the two ensembles, now built from all four.
Run the core track (00-10) first.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

MODEL = "Explainable Boosting"
FOLD_JOBS = 5

In [ ]:
from anc8 import models as zoo
class Space:
    def __init__(self): self.rows = []
    def suggest_float(self, n, lo, hi, log=False, step=None): self.rows.append((n, lo, hi, "log" if log else "")); return lo
    def suggest_int(self, n, lo, hi, step=1, log=False): self.rows.append((n, lo, hi, f"step {step}")); return lo
    def suggest_categorical(self, n, ch): self.rows.append((n, ch, "", "one of")); return ch[0]
sp = Space(); zoo.suggest(sp, MODEL)
print(pd.DataFrame(sp.rows, columns=["setting", "from", "to", ""]).to_string(index=False))
print(f"\n{tuning.TRIALS[MODEL]} trials per predictor set and rung")

In [ ]:
for s in lib.SETS:
    for r in lib.RUNGS:
        job = lib.Job("tune", s, r, MODEL, MODEL)
        if not lib.is_done(P.results, job):
            lib.run(job, P.results, fold_jobs=FOLD_JOBS)
        for method in lib.CALIBRATIONS:
            cj = lib.Job("calib", s, r, lib.calibrated_name(MODEL, method), MODEL)
            if not lib.is_done(P.results, cj):
                lib.run(cj, P.results)
for job in lib.lineup_jobs("four"):
    if not lib.is_done(P.results, job):
        lib.run(job, P.results)
repro.run_script("fit_library_models.py", "--root", P.results, "--lineup", "four")

In [ ]:
repro.compare_cells(P.results, [MODEL] + [lib.calibrated_name(MODEL, c) for c in lib.CALIBRATIONS])
res = repro.compare_cells(P.results, [lib.ensemble_name("stack", "four"), lib.ensemble_name("cumlink", "four")])
print(repro.compare_cells(P.results, [MODEL]).pivot_table(index="set", columns="rung", values="held_AP").round(3).to_string())
repro.report()